Document Structure

In [5]:
from langchain_core.documents import Document

In [6]:
doc = Document(
    page_content = "This is test content",
    metadata = {
        "source": "test",
        "createdBy": "Deepak Bhatt"
    }
)

doc

Document(metadata={'source': 'test', 'createdBy': 'Deepak Bhatt'}, page_content='This is test content')

**Loading**

In [7]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.document_loaders import DirectoryLoader

In [8]:
loader = CSVLoader(file_path="../data/text_files/customers-100.csv")
data = loader.load()

In [9]:
data

[Document(metadata={'source': '../data/text_files/customers-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
 Document(metadata={'source': '../data/text_files/customers-100.csv', 'row': 1}, page_content='Index: 2\nCustomer Id: 1Ef7b82A4CAAD10\nFirst Name: Preston\nLast Name: Lozano\nCompany: Vega-Gentry\nCity: East Jimmychester\nCountry: Djibouti\nPhone 1: 5153435776\nPhone 2: 686-620-1820x944\nEmail: vmata@colon.com\nSubscription Date: 2021-04-23\nWebsite: http://www.hobbs.com/'),
 Document(metadata={'source': '../data/text_files/customers-100.csv', 'row': 2}, page_content='Index: 3\nCustomer Id: 6F94879bDAfE5a6\nFirst Name: Roy\nLast Name: Berry\nCompany: Murillo-Perry\nCity: Isabelborough\nCountry: Antigua a

In [10]:
#To load all csvs in a directory

dir_loader = DirectoryLoader(
    "../data",
    glob="**/*.csv",
    loader_cls=CSVLoader
)

csvload = dir_loader.load()
csvload

[Document(metadata={'source': '..\\data\\text_files\\customers-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
 Document(metadata={'source': '..\\data\\text_files\\customers-100.csv', 'row': 1}, page_content='Index: 2\nCustomer Id: 1Ef7b82A4CAAD10\nFirst Name: Preston\nLast Name: Lozano\nCompany: Vega-Gentry\nCity: East Jimmychester\nCountry: Djibouti\nPhone 1: 5153435776\nPhone 2: 686-620-1820x944\nEmail: vmata@colon.com\nSubscription Date: 2021-04-23\nWebsite: http://www.hobbs.com/'),
 Document(metadata={'source': '..\\data\\text_files\\customers-100.csv', 'row': 2}, page_content='Index: 3\nCustomer Id: 6F94879bDAfE5a6\nFirst Name: Roy\nLast Name: Berry\nCompany: Murillo-Perry\nCity: Isabelborough\nCountry: 

In [11]:
type(csvload)
type(csvload[0])

langchain_core.documents.base.Document

**Splitting/Chunking**

In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [13]:
def process_all_pdfs(directory_path):
    all_documents = []
    pdf_dir = Path(directory_path)

    #Find all pdf files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} files")

    for pdf in pdf_files:
        print(f"File Name:{pdf.name}")

        try:
            loader = PyPDFLoader(str(pdf))

            documents = loader.load()

            for doc in documents:
                doc.metadata['file_name']= pdf.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print("Error")

    print(f"\nTotal documents loaded:{len(all_documents)}")
    return all_documents


all_pdfs= process_all_pdfs("../data")



Found 2 files
File Name:Get_Started_With_Smallpdf.pdf
  ✓ Loaded 1 pages
File Name:The-war-of-art-by-Robert-Pressfield.pdf
  ✓ Loaded 159 pages

Total documents loaded:160


In [14]:
all_pdfs

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2020-10-14T17:08:10+02:00', 'moddate': '2020-10-14T17:08:10+02:00', 'trapped': '/False', 'source': '..\\data\\Get_Started_With_Smallpdf.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'Get_Started_With_Smallpdf.pdf', 'file_type': 'pdf'}, page_content='Welcome to Smallpdf\nDigital Documents—All In One Place\nAccess Files Anytime, Anywhere \nEnhance Documents in One Click \nCollaborate With Others \nWith the new Smallpdf experience, you can \nfreely upload, organize, and share digital \ndocuments. When you enable the ‘Storage’ \noption, we’ll also store all processed files here. \nYou can access files stored on Smallpdf from \nyour computer, phone, or tablet. We’ll also \nsync files from the Smallpdf Mobile App to our \nonline portal\nWhen you right-click on a file, we’ll present \nyou with an array of options to convert, \ncompress, or modify it. \n

**documents = one PDF**
all_documents = all PDFs
all_pdfs = returned result of function

Chunking

In [15]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[10].page_content[:200]}...")
        print(f"Metadata: {split_docs[10].metadata}")
    
    return split_docs

In [16]:
chunks=split_documents(all_pdfs)
chunks

Split 160 documents into 239 chunks

Example chunk:
Content: acting in the face of fear and failure—no excuses, no bullshit. 
And best of all, Steve’s brilliant insight that ﬁrst, last, and 
always, the professional focuses on mastery of the craft.
Book Three, ...
Metadata: {'producer': 'Mac OS X 10.6.4 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20100929182026Z00'00'", 'title': '**Final_The War of Art_6x9_Final**', 'author': 'Mike Bertoldo', 'moddate': "D:20100929182026Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\The-war-of-art-by-Robert-Pressfield.pdf', 'total_pages': 159, 'page': 7, 'page_label': '8', 'file_name': 'The-war-of-art-by-Robert-Pressfield.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2020-10-14T17:08:10+02:00', 'moddate': '2020-10-14T17:08:10+02:00', 'trapped': '/False', 'source': '..\\data\\Get_Started_With_Smallpdf.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'Get_Started_With_Smallpdf.pdf', 'file_type': 'pdf'}, page_content='Welcome to Smallpdf\nDigital Documents—All In One Place\nAccess Files Anytime, Anywhere \nEnhance Documents in One Click \nCollaborate With Others \nWith the new Smallpdf experience, you can \nfreely upload, organize, and share digital \ndocuments. When you enable the ‘Storage’ \noption, we’ll also store all processed files here. \nYou can access files stored on Smallpdf from \nyour computer, phone, or tablet. We’ll also \nsync files from the Smallpdf Mobile App to our \nonline portal\nWhen you right-click on a file, we’ll present \nyou with an array of options to convert, \ncompress, or modify it. \n

Embedding and Vector Store DB

In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
class Student:
    def __init__(self,name,age):
        self.name=name
        self.age=age

s1 = Student('Deepak',22)

print(s1.name)
print(s1.age)

Deepak
22


In [19]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2405.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


VectorStore

In [20]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 239


In [21]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2020-10-14T17:08:10+02:00', 'moddate': '2020-10-14T17:08:10+02:00', 'trapped': '/False', 'source': '..\\data\\Get_Started_With_Smallpdf.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'Get_Started_With_Smallpdf.pdf', 'file_type': 'pdf'}, page_content='Welcome to Smallpdf\nDigital Documents—All In One Place\nAccess Files Anytime, Anywhere \nEnhance Documents in One Click \nCollaborate With Others \nWith the new Smallpdf experience, you can \nfreely upload, organize, and share digital \ndocuments. When you enable the ‘Storage’ \noption, we’ll also store all processed files here. \nYou can access files stored on Smallpdf from \nyour computer, phone, or tablet. We’ll also \nsync files from the Smallpdf Mobile App to our \nonline portal\nWhen you right-click on a file, we’ll present \nyou with an array of options to convert, \ncompress, or modify it. \n

In [22]:
# Generate embeddings for the document chunks
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 239 texts...


Batches: 100%|██████████| 8/8 [00:06<00:00,  1.21it/s]


Generated embeddings with shape: (239, 384)
Adding 239 documents to vector store...
Successfully added 239 documents to vector store
Total documents in collection: 478


In [23]:
embeddings.shape

(239, 384)

Retriever Pipeline From VectorStore

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("What did Krishna say to Arjuna?")

Retrieving documents for query: 'What did Krishna say to Arjuna?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c21e702a_231',
  'content': 'THE FRUITS OF OUR LABOR\n______________________________\nWhen Krishna instructed Arjuna that we have a right to our \nlabor but not to the fruits of our labor, he was counseling the \nwarrior to act territorially, not hierarchically. We must do our \nwork for its own sake, not for fortune or attention or applause.\nThen there’s the third way proffered by the Lord of \nDiscipline, which is beyond both hierarchy and territory. That \nis to do the work and give it to Him. Do it as an offering to \nGod.\nGive the act to me.\nPurged of hope and ego,\nFix your attention on the soul.\nAct and do for me.\nThe work comes from heaven anyway. Why not give it back?\nTo labor in this way, The Bhagavad-Gita tells us, is a form of \nmeditation and a supreme species of spiritual devotion. It also, \nI believe, conforms most closely to Higher Reality. In fact, we \nare servants of the Mystery. We were put here on earth to act \nas agents of the Inﬁnite, to brin

In [27]:
rag_retriever.retrieve("How to fight resistance?")

Retrieving documents for query: 'How to fight resistance?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.43it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bfcab3eb_25',
  'content': 'BOOK ONE\n___________\nRESISTANCE\nDeﬁning the Enemy',
  'metadata': {'doc_index': 25,
   'total_pages': 159,
   'creator': 'Pages',
   'page': 15,
   'source': '..\\data\\The-war-of-art-by-Robert-Pressfield.pdf',
   'aapl:keywords': '[]',
   'author': 'Mike Bertoldo',
   'file_name': 'The-war-of-art-by-Robert-Pressfield.pdf',
   'creationdate': "D:20100929182026Z00'00'",
   'producer': 'Mac OS X 10.6.4 Quartz PDFContext',
   'title': '**Final_The War of Art_6x9_Final**',
   'keywords': '',
   'file_type': 'pdf',
   'content_length': 49,
   'page_label': '16',
   'moddate': "D:20100929182026Z00'00'"},
  'similarity_score': 0.3470081686973572,
  'distance': 0.6529918313026428,
  'rank': 1},
 {'id': 'doc_95580bb8_25',
  'content': 'BOOK ONE\n___________\nRESISTANCE\nDeﬁning the Enemy',
  'metadata': {'page_label': '16',
   'author': 'Mike Bertoldo',
   'doc_index': 25,
   'content_length': 49,
   'page': 15,
   'file_name': 'The-war-of-art-by-Robe

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [33]:
answer=rag_simple("What is resistance?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is resistance?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Resistance is a force of nature that acts objectively and impersonally, using various forms of deception to prevent individuals from doing their work.
